In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm, beta as beta_dist, gamma as gamma_dist


In [2]:
def precision_poisson_stringer(
    sample_s: pd.DataFrame,
    EE: float,
    SI: float,
    cl: float,
):

    errors = (
        sample_s.loc[
            (sample_s["E"] != 0),
            ["ER"],
        ]
        .copy()
        .sort_values("ER", ascending=False)
        .reset_index(drop=True)
    )

    taints = errors["ER"].to_numpy(dtype=float)

    basic_rf = gamma_dist.ppf(q=cl, a=1, scale=1)

    BP = SI * basic_rf

    ranks = np.arange(1, len(taints) + 1)

    incremental_factors = (
        gamma_dist.ppf(q=cl, a=ranks + 1, scale=1) - gamma_dist.ppf(q=cl, a=ranks, scale=1) - 1
    )

    IA = SI * np.dot(incremental_factors, taints)

    SE = BP + IA
    ULE = EE + SE
    VAR = 0

    return SE, VAR, ULE

In [3]:
from scipy.stats import beta as beta_dist
import numpy as np
import pandas as pd


def precision_binomial_stringer(
    sample_s: pd.DataFrame,
    EE: float,
    SI: float,
    cl: float,
):

    errors = (
        sample_s.loc[
            (sample_s["E"] != 0),
            ["ER"],
        ]
        .copy()
        .sort_values("ER", ascending=False)
        .reset_index(drop=True)
    )

    taints = errors["ER"].to_numpy(dtype=float)

    n = len(sample_s)

    basic_rf = n * beta_dist.ppf(q=cl, a=1, b=n,)

    BP = SI * basic_rf

    ranks = np.arange(1, len(taints) + 1)

    incremental_factors = (
        n * (
            beta_dist.ppf(q=cl, a=ranks + 1, b=n - ranks,) - 
            beta_dist.ppf( q=cl, a=ranks, b=n - ranks + 1, )
        )
        - 1
    )
    incremental_factors = np.where(incremental_factors==np.nan, n, incremental_factors)

    IA = SI * np.dot(incremental_factors, taints)

    SE = BP + IA
    ULE = EE + SE
    VAR = 0

    return SE, VAR, ULE

In [4]:
def precision_binomial_stringer(
    sample_s: pd.DataFrame,
    EE: float,
    BV: float,
    cl: float,
    n: int,
):
    errors = (
            sample_s.loc[
                (sample_s["E"] != 0),
                ["ER"],
            ]
            .copy()
            .sort_values("ER", ascending=False)
            .reset_index(drop=True)
        )
    
    taints = errors["ER"].to_numpy(dtype=float)

    BP = (1 - (1 - cl) ** (1 / n))

    ranks = np.arange(1, len(taints) + 1)

    # p_k^U = Beta^-1(cl; k+1, n-k) is undefined at k=n (every sampled item
    # tainted): shape2 = n-k = 0. The analytical limit is 1 exactly, since an
    # error rate cannot exceed 1 -- substitute it there instead of letting it
    # evaluate to NaN. (This can only occur at the single largest rank, and
    # only when every sampled item is tainted; the second term below never
    # hits shape2=0 since n-(k-1) >= 1 for all k in [1, n].)
    b_first = n - ranks
    first_term = np.where(
        b_first == 0,
        1.0,
        beta_dist.ppf(q=cl, a=ranks + 1, b=np.where(b_first == 0, 1, b_first)),
    )
    incremental_factors = first_term - beta_dist.ppf(q=cl, a=ranks, b=n - (ranks-1))

    IA = np.dot(incremental_factors, taints)

    SE = (BP + IA) * BV
    ULE = EE + SE
    VAR = 0

    return SE, VAR, ULE

In [49]:
gamma_dist.ppf(q=0.9, a=1, scale=1)*50

np.float64(115.12925464970229)

In [50]:
gamma_dist.ppf(q=0.9, a=1, scale=50)

np.float64(115.12925464970229)

In [40]:
1-(1-0.9)**(1/100)

0.02276277904418933

In [48]:
beta_dist.ppf(q=0.9, a=1, b=100)

np.float64(0.022762779044189316)

In [34]:
ranks = np.arange(1, 1000 + 1)
cl = 0.9
first_term = gamma_dist.ppf(q=cl, a=ranks + 1, scale=1)
second_term = gamma_dist.ppf(q=cl, a=ranks, scale=1)
incremental_factors = (
        first_term - second_term - 1
    )
incremental_factors

array([0.58713508, 0.43260017, 0.35846273, 0.31280652, 0.28108431,
       0.25739821, 0.23884236, 0.22379708, 0.21127875, 0.20065088,
       0.19148097, 0.18346349, 0.17637564, 0.1700506 , 0.16436067,
       0.15920622, 0.15450819, 0.15020297, 0.14623869, 0.14257262,
       0.13916914, 0.13599833, 0.13303483, 0.13025699, 0.1276462 ,
       0.12518637, 0.12286348, 0.12066529, 0.11858101, 0.11660114,
       0.11471725, 0.11292181, 0.11120813, 0.10957017, 0.10800252,
       0.10650029, 0.10505904, 0.10367478, 0.10234383, 0.10106285,
       0.09982881, 0.09863889, 0.09749054, 0.09638138, 0.09530923,
       0.09427209, 0.09326809, 0.0922955 , 0.09135271, 0.09043824,
       0.08955069, 0.08868877, 0.08785127, 0.08703706, 0.08624507,
       0.08547432, 0.08472387, 0.08399284, 0.08328042, 0.08258582,
       0.08190832, 0.08124723, 0.08060188, 0.07997168, 0.07935603,
       0.07875438, 0.07816621, 0.07759103, 0.07702836, 0.07647776,
       0.0759388 , 0.07541107, 0.0748942 , 0.07438781, 0.07389

In [36]:
ranks = np.arange(1, 1000 + 1)
n= 1000
cl=0.9
first_term = np.where(
        ranks < n,
        n * beta_dist.ppf(q=cl, a=ranks + 1, b=n - ranks,),
        n,
    )
second_term = n * beta_dist.ppf(q=cl, a=ranks, b=n - ranks + 1)
incremental_factors = first_term - second_term
incremental_factors

array([1.58416822, 1.42938097, 1.35500997, 1.30914063, 1.27722213,
       1.25335343, 1.23462623, 1.21941904, 1.20674682, 1.19597199,
       1.18666118, 1.17850814, 1.17128948, 1.16483792, 1.15902534,
       1.15375174, 1.14893778, 1.14451959, 1.14044507, 1.13667128,
       1.13316243, 1.12988844, 1.12682381, 1.12394676, 1.12123858,
       1.11868305, 1.11626607, 1.1139753 , 1.1117999 , 1.10973028,
       1.10775793, 1.10587529, 1.10407558, 1.10235273, 1.10070126,
       1.09911625, 1.09759323, 1.09612813, 1.09471727, 1.09335726,
       1.09204502, 1.09077773, 1.08955279, 1.08836781, 1.08722057,
       1.08610904, 1.08503133, 1.0839857 , 1.08297051, 1.08198425,
       1.08102552, 1.08009301, 1.07918547, 1.07830177, 1.07744084,
       1.07660165, 1.07578327, 1.0749848 , 1.07420542, 1.07344432,
       1.07270077, 1.07197406, 1.07126354, 1.07056857, 1.06988856,
       1.06922296, 1.06857122, 1.06793284, 1.06730735, 1.06669428,
       1.0660932 , 1.06550371, 1.0649254 , 1.06435791, 1.06380

In [ ]:
def precision_poisson_stringer(
    sample_s: pd.DataFrame,
    EE: float,
    SI: float,
    cl: float,
):
    
    taints = sample_s["ER"].to_numpy(dtype=float)
    taints = np.sort(taints[taints > 0])[::-1]

    basic_rf = gamma_dist.ppf(q=cl, a=1, scale=1,)

    BP = SI * basic_rf

    ranks = np.arange(1, len(taints) + 1)

    current_rf = gamma_dist.ppf(q=cl, a=ranks + 1, scale=1,)
    previous_rf = gamma_dist.ppf(q=cl, a=ranks,scale=1,)

    incremental_factors = current_rf - previous_rf

    IA = SI * np.dot(incremental_factors, taints)

    ULE = BP + IA
    SE = ULE - EE
    VAR = 0

    if SE < 0:
        raise ValueError(
            f"Negative Poisson Stringer precision ({SE}). "
            "Check EE, SI, cl, and the definition of ER."
        )

    return SE, VAR, ULE

def precision_binomial_stringer(
    sample_s: pd.DataFrame,
    EE: float,
    SI: float,
    cl: float,
):
    n = len(sample_s)
    taints = sample_s["ER"].to_numpy(dtype=float)
    taints = np.sort(taints[taints > 0])[::-1]

    basic_rf = n * beta_dist.ppf(q=cl, a=1, b=n,)

    BP = SI * basic_rf

    ranks = np.arange(1, len(taints) + 1)

    # CF_k = n * Beta^-1(cl; k+1, n-k), k < n
    # CF_n = n
    current_rf = np.empty(len(ranks), dtype=float)

    mask = ranks < n

    current_rf[mask] = n * beta_dist.ppf(q=cl, a=ranks[mask] + 1, b=n - ranks[mask],)
    current_rf[~mask] = n

    previous_rf = n * beta_dist.ppf(q=cl, a=ranks, b=n - ranks + 1,)

    incremental_factors = current_rf - previous_rf

    IA = SI * np.dot(incremental_factors, taints)

    ULE = BP + IA
    SE = ULE - EE
    VAR = 0

    if SE < 0:
        raise ValueError(
            f"Negative Binomial Stringer precision ({SE}). "
            "Check EE, SI, cl, and the definition of ER."
        )

    return SE, VAR, ULE